# Liquid MCDM Lab · VS Code + Colab Pro entry

Open this notebook in VS Code, choose **Select Kernel → Colab → Auto Connect** using Google's official `google.colab` extension, and sign in through Google OAuth. Select a Colab NVIDIA GPU runtime for all execution. It may consume Pro compute units; actual allocation is measured below.

The kernel runs in Colab: local files and `.env` are not automatically present. Upload the already prepared `dist/colab-source.zip` through the extension's runtime file workflow to `/content/colab-source.zip`. The bootstrap cell extracts each archive version into its own hash-named remote source directory, preserving earlier outputs.

The bootstrap cell installs the locked tooling into the remote runtime. Restart the kernel if required. If a Git remote is later created, clone and checkout the recorded commit instead. Never upload `.env` or paste API keys into notebook cells.

Use Colab Secrets (`HF_TOKEN`, `WANDB_API_KEY`) only when authentication is actually needed. CPU smoke uses W&B offline and needs neither. This launcher has no model-loading/training cell.


Run the cells below in order on the already-connected Colab GPU kernel. They bootstrap the uploaded source and run the complete remote acceptance suite. The final report explicitly leaves model training unvalidated. No local package install is needed.


In [ ]:
"""Execution policy: workloads require a managed remote NVIDIA GPU runtime.

This standard-library-only module is also embedded in notebook preflight cells,
so hardware checks work before any package or project installation.
"""
import os
from pathlib import Path
import shutil
import subprocess


GPU_SETUP_HELP = (
    'In browser Colab select Runtime > Change runtime type > Hardware accelerator '
    '> an available NVIDIA GPU, then reconnect. In VS Code select a Colab server '
    'configured with an NVIDIA GPU and reconnect this notebook to that server. '
    'Run the preflight in the same runtime used for nbconvert. '
    'Do not try to fix GPU allocation with pip or install NVIDIA drivers in Colab.'
)


def find_nvidia_smi():
    """Resolve installed driver tooling, including common paths omitted from PATH."""
    executable = shutil.which('nvidia-smi')
    if executable:
        return executable
    for candidate in ('/usr/bin/nvidia-smi', '/usr/local/nvidia/bin/nvidia-smi'):
        if Path(candidate).is_file() and os.access(candidate, os.X_OK):
            return candidate
    return None


def require_colab_gpu():
    managed = bool(os.environ.get('COLAB_RELEASE_TAG') or
                   (Path('/content').is_dir() and os.environ.get('COLAB_BACKEND_VERSION')))
    if not managed:
        raise RuntimeError('Colab-only execution: connect a managed Colab GPU kernel; do not run locally.')
    return query_gpu('Colab', GPU_SETUP_HELP)


def require_remote_gpu():
    """Operational guard, not a security attestation: environment flags are spoofable."""
    if os.environ.get('JOB_ID'):
        accelerator = os.environ.get('ACCELERATOR', '').strip()
        if accelerator.lower() in ('', 'none'):
            raise RuntimeError('Hugging Face Jobs requires an NVIDIA GPU flavor, such as t4-small.')
        gpu = query_gpu('Hugging Face Jobs', 'Select an NVIDIA GPU Jobs flavor and retry.')
        return {'provider': 'huggingface_jobs', 'job_id': os.environ['JOB_ID'],
                'accelerator': accelerator, 'gpu': gpu}
    return {'provider': 'colab', 'gpu': require_colab_gpu()}


def query_gpu(provider, help_text):
    executable = find_nvidia_smi()
    if executable is None:
        raise RuntimeError(
            'NVIDIA GPU tooling is unavailable: nvidia-smi was not found in PATH or '
            'common driver locations. This session may be CPU/TPU or lack an attached '
            'NVIDIA GPU; no Python dependency installation has been attempted by this check. '
            + help_text
        )
    try:
        result = subprocess.run(
            [executable, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=15,
        )
    except (OSError, subprocess.TimeoutExpired) as exc:
        raise RuntimeError(f'Could not query the {provider} NVIDIA GPU. ' + help_text) from exc
    if result.returncode or not result.stdout.strip():
        raise RuntimeError('nvidia-smi did not report an available NVIDIA GPU. ' + help_text)
    return result.stdout.strip()

import sys
print(require_colab_gpu())


In [ ]:
# Runs in the connected Colab VM only. Upload the prepared source ZIP first.
import zipfile
import hashlib
SOURCE_ZIP = Path("/content/colab-source.zip")
assert SOURCE_ZIP.exists(), "Upload the prepared dist/colab-source.zip to /content using the Colab extension."
archive_hash = hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest()
SNAPSHOT_ROOT = Path("/content/liquid-mcdm-snapshots") / archive_hash
REMOTE_REPO = SNAPSHOT_ROOT / "decision-engines"
receipt = SNAPSHOT_ROOT / "source-archive.sha256"
if receipt.exists():
    assert receipt.read_text().strip() == archive_hash, "Source snapshot receipt mismatch"
    assert (REMOTE_REPO / "pyproject.toml").exists(), "Source snapshot is incomplete"
else:
    assert not SNAPSHOT_ROOT.exists(), "Incomplete prior extraction: choose a fresh snapshot directory before retrying"
    SNAPSHOT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        for name in archive.namelist():
            destination = (SNAPSHOT_ROOT / name).resolve()
            assert destination.is_relative_to(REMOTE_REPO), "Unexpected archive path"
        archive.extractall(SNAPSHOT_ROOT)
    receipt.write_text(archive_hash + "\n")
os.environ["LIQUID_MCDM_REPO"] = str(REMOTE_REPO)
os.chdir(REMOTE_REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REMOTE_REPO / "requirements-cpu.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REMOTE_REPO)], check=True)
print("Remote source:", REMOTE_REPO)
print("Restart this kernel if requested by pip; keep the same Colab connection.")


In [ ]:
# All tests, generation, tokenizer downloads and W&B offline work stay on Colab.
# This runner never downloads model weights or starts training.
subprocess.run([sys.executable, str(REMOTE_REPO / "scripts/verify_colab.py")], cwd=REMOTE_REPO, check=True)


Continue in order:

1. `notebooks/00_environment_check.ipynb`
2. `notebooks/01_mcdm_data_and_baselines.ipynb`
3. `notebooks/02_d1_3b_lora_colab.ipynb` — gated preparation shell
4. `notebooks/03_evaluate_and_compare.ipynb`

See `README.md` for persistence, safe authentication and the next GPU approval boundary. A connected Colab kernel is not evidence that the model recipe has passed forward/backward or reload checks.